In [2]:
import numpy as np
import matplotlib.pyplot as plt 
from windfreak import SynthHD
import time
from datetime import datetime
import nidaqmx
from nidaqmx.constants import Edge
from nidaqmx.constants import AcquisitionType
from pyrpl import Pyrpl
# import pyvisa
import h5py
import os
from Prayas_codes import *

%matplotlib qt

ModuleNotFoundError: No module named 'windfreak'

In [41]:
# TTL SETUP USING DAQ CARD

sampling_rate_out = 5000    # Sampling rate for the TTL Output   # This is the maximum output sample rate for USB6003
samps_out = 4               # ALWAYS EVEN FOR 50% DUTY CYCLE. Chosen in a way to fix modulation frequency   # Modulation frequency at 1250Hz for USB6003 with 4 samps out
mod_freq = np.round(sampling_rate_out/samps_out,2) # Modulation frequency calculated from samples chosen   # 5000/4 = 1250Hz
delay_samples = 0           # ALWAYS LESS THAN samps_out. Phase difference between TTL to LockIn and MW Switch in terms of samples

rp_scan_freq = np.linspace(mod_freq-2, mod_freq+2,5)

if delay_samples > samps_out: delay_samples = delay_samples % samps_out # ensuring delay_samples < samps_out


# Defining DAQ Card control object
ao_task = nidaqmx.Task()   # nidaq Task only for generating TTL outputs from DAQ
#ao_task.ao_channels.add_ao_voltage_chan('Dev7/ao0')
ao_task.ao_channels.add_ao_voltage_chan('Dev7/ao1')  #no need for two TTLs in red pitaya case

ao_task.timing.cfg_samp_clk_timing(sampling_rate_out, sample_mode=AcquisitionType.CONTINUOUS, samps_per_chan=samps_out)  # Configuring the clock timing


# Creating the TTL pulse according to above settings
k1 = 5*np.ones(int(samps_out/2))    # generates an array of 5s for half the samples
k1 = np.append(k1, np.zeros(int(samps_out/2)))  # generates an array of 0s for half the samples
# if delay_samples < int(samps_out/2): 
#     k2 = np.zeros(delay_samples)   # generates a TTL signal in the same way with little phase shift/delay from k1 array
#     k2 = np.append(k2, 5*np.ones(int(samps_out/2)))
#     k2 = np.append(k2, np.zeros(int(samps_out/2) - delay_samples))
# else:
#     k2 = 5*np.ones(delay_samples - int(samps_out/2))
#     k2 = np.append(k2, np.zeros(int(samps_out/2)))
#     k2 = np.append(k2, 5*np.ones(int(samps_out) - delay_samples))

# Beginning the TTL
ao_task.write(np.array([k1]))

# print(k1)
# print(k2)


68

In [42]:
# SETTING MW SOURCE CONTROL

synth = SynthHD("COM3")  # Windfreak
synth.init()
rf = synth[1]
rf.enable = False

p = Pyrpl(config = '', hostname = '192.168.0.35', gui = False)
r = p.rp

iq0 = r.iq0
iq1 = r.iq1
s = r.scope

iq0.output_direct = 'off'
iq1.output_direct = 'off'

s.input1 = 'iq0'
s.input2 = 'iq1'
s.trigger_source = 'immediately'
s.threshold = 0.1
s.hysteresis = 0.01
s.average = True
s.continuous = True

In [43]:
# LIA/PS-1 META DATA BLOCK

def smallGenMeta(rf, current_preamp = 1, preamp_gain = 6, mw_switch = 0, mw_gen = 0, laser_C1 = 8.75, photodetector = 0,
                 mw_amp = 0, time_constant = 0.1, mod_freq = 2000, start_freq = 2840e6, stop_freq = 2900e6, 
                 n_points = 200, sweep_runs = 1, wait_time = 0.5, comments = '', sample = 'MPCVD-1',
                 sampling_rate = 5000, sweep_time = time.strftime("%H:%M:%S", time.gmtime(0.5*200)), 
                 total_time = time.strftime("%H:%M:%S", time.gmtime(0.5*200)), lockin_gain = 10, shift = 0, samps_out = 50):

    FILTER_SLOPE = 6                       # dB/Octave (basic settings on LIA/PS-1)

    PHOTODETECTORS = ['DET36A2', 'DET100A/M', 'DET100A2','A-CUBES3000-03']   # we are using A-CUBES3000-03 for now (APD)
    CURRENT_PREAMPS = ['None', '1641', '1212']
    SWITCHES = ['ZYSWA-2-50DR+', 'ZASW-2-50DR+']  # ZYSWA-2-50DR+
    MW_AMPS = ['None', 'ZVE-3W-83+', 'ZHL-4240+']  #ZVE-3W-83+
    MW_GENERATORS = ['Windfreak SynthHD Pro V1', 'Signal Hound VSG60A']  #Windfreak

    # EXPERIMENT SETTINGS

    gen_set = {}
    gen_set['Date Time'] = datetime.now().isoformat()
    gen_set['MW Source'] = MW_GENERATORS[mw_gen]
    gen_set['MW Switch'] = SWITCHES[mw_switch]
    gen_set['MW PowerAmp'] = MW_AMPS[mw_amp]
    gen_set['Photo Diode'] = PHOTODETECTORS[photodetector]
    gen_set['Current Preamp'] = CURRENT_PREAMPS[current_preamp]
    gen_set['Lock-In Amp'] = 'RedPitaya'
    gen_set['TTL Source'] = 'NI USB6003'  # input bandwidth of 300 kHz
    gen_set['Sample'] = sample

    # MEASUREMENT SETTINGS

    meas_set = {}
    meas_set['Laser C1'] = laser_C1
    meas_set['Preamp Gain (V per A)'] = 10 ** preamp_gain  #This is given in dB so 10**gain
    meas_set['MW Power (dBm)'] = rf.power
    meas_set['Modulation Frequency (Hz)'] = mod_freq
    meas_set['DC Sampling Rate (Hz)'] = sampling_rate
    meas_set['Time Constant (s)'] = time_constant # default is 100ms
    meas_set['Filter Slope dB/Octave'] = FILTER_SLOPE  # 6dB/octave or 12dB/octave
    meas_set['MW Frequency Sweep (MHz)'] = np.array([start_freq/1e6 , stop_freq/1e6 ])
    meas_set['No. of Sweep Points'] = n_points  # number of points to plot
    meas_set['No. of Sweep Runs'] = sweep_runs  # number of sweep runs to do, default is a single run
    meas_set['Lock-In Wait Time (s)'] = wait_time  # default is 500ms
    meas_set['Lock-In Gain'] = lockin_gain
    meas_set['Lock-In True Gain'] = lockin_gain  # have to figure this one out from datasheet  # sensitivity is set to 300mV so total gain is (ultra stable gain)*(1V/300mV)*sqrt(2) for rms to amplitude
    meas_set['Sweep time'] = sweep_time
    meas_set['Total time'] = total_time
    meas_set['MW Switch TTL ahead of Lock-In TTL by samples'] = shift   # number of samples the MW switch TTL is ahead of Lock-In TTL
    meas_set['Samples in one TTL cycle'] = samps_out
    #meas_set['Lock-In TTL Channel on DAQ'] = 'AO0'
    meas_set['MW Switch TTL Channel on DAQ'] = 'AO1'
    meas_set['Red Pitaya Frequency Sweep'] = np.linspace(mod_freq-2, mod_freq+2,5)  # This is for sweeping over frequencies to check which is best for red pitaya
    meas_set['Comments'] = comments

    # BLOCK DIAGRAM DATA

    block_set = {}
    block_set['Microwave'] = f"MW Source ({gen_set['MW Source']}) --> MW Switch ({gen_set['MW Switch']}) --> Antenna + Sample"
    block_set['Optics'] = f"Laser (532 nm) --> Gold Mirror --> Antenna + Sample --> Objective lens --> Detector ({gen_set['Photo Diode']})"
    block_set['Measurement'] = f"Detector ({gen_set['Photo Diode']}) --> Photodiode Amplifier ({gen_set['Current Preamp']}) --> DAQ Card (USB-6343) + Lock-In Amp ({gen_set['Lock-In Amp']}) + Oscilloscope (TDS 2022B) --> DAQ Card (USB-6343)"
    #block_set['TTL Lock-In'] = f"Signal Generator ({gen_set['TTL Source']} Channel {meas_set['Lock-In TTL Channel on DAQ']}) --> Lock-In Amp ({gen_set['Lock-In Amp']})"
    block_set['TTL MW Switch'] = f"Signal Generator ({gen_set['TTL Source']} Channel {meas_set['MW Switch TTL Channel on DAQ']}) --> MW Switch ({gen_set['MW Switch']})"

    # CREATING THE META DATA DICTIONARY 
    
    meta_dict = {}
    meta_dict['General Filler'] = '=====================General Settings==================='
    meta_dict['General Settings'] = gen_set
    meta_dict['Measurement Filler'] = '===============Measurement Settings================='
    meta_dict['Measurement Settings'] = meas_set
    meta_dict['Block Filler'] = '==========================Block Info======================'
    meta_dict['Block_info'] = block_set
    
    return meta_dict



In [44]:
# CREATING DATA FOLDER IN DRIVE D

dir = 'E:/Large Area ODMR/'                                     # Directory which holds all data
sample = 'MPCVD/SAMPLE 1'                                       # Specific sample folder
dir = dir + sample + '/'
fold = time.strftime("%Y%m%d")                                  # A new folder inside the sample older corresponding to the date
# fold = ''
pathname = dir + fold                                           # defining the path

if pathname.endswith('/'): pathname = pathname
else: pathname = pathname + '/'

path1 = File_Manipulator(pathname)                              # creating folder manipulation object
path1.make_it()                                                 # creates the folder if it doesn't exist

# print(path1.pathname)

Path exists.


In [45]:
# CREATING DATA STORAGE ARRAYS AND TEST GENERATING META DATA


rf.power = 7           # generated RF power (dBm) ***KEEP BELOW 0 WHEN AMPLIFIER IS CONNECTED***
start_freq = 2840.e6    # Enter start Frequency (Hz)
stop_freq = 2900.e6     # Enter stop Frequency (Hz)
freq_points = 50        # Enter number of points
sweep_runs = len(rp_scan_freq)          # Enter the number of runs
laser_C1 = 8.75         # laser C1 set on laser power supply (mW)
preamp_gain = 0         # Preamp gain (10^{preamp_gain} V/A)
current_preamp = 0      # (0-None, 1-1641, 2-PDA200C)
mw_amp = 1              # (0-None, 1-ZVE-3W-83+, 2-ZHL-4240+)
mw_gen = 0              # (0-Windfreak SynthHD, 1-VSG60A)
mw_switch = 0           # (0-ZYSWA-2-50DR+, 1-ZASW-2-50DR+)
photodetector = 3       # (0-DET36A2, 1-DET100A/M, 2-DET100A2, 3-A-CUBES3000-03)
sample = 'MPCVD-1'      # Name of the sample used with any specifications
termination_flag = 0    # To indicate if the data acquisition loop was force-stopped
lockin_gain = 1       # Gain setting on Lockin Amp
time_constant = 0.1     # Time constant set on Lockin Amp  # check the time constant on lockin # 10ms right now
plot_delay = 0.1        # Delay needed by matplotlib
wait_time = 10*time_constant         # Wait time for the lock in reading
sampling_rate_in = 5000  # Sampling rate for the DC reading  # analog input sampling rate is 1e5 S/s so should work
samps_per_chan = int(sampling_rate_in*(wait_time - plot_delay))   #1e5*(10*0.1 - 0.1) = 1e5*0.9
#lockin_offset = -0.088 # check maybe just offset in the lockin used at the time, shouldn't matter much

#comments = 'Dark Current = 2.84 nA.'   # Always include the dark current measured before the experiment in the comments


dc_level_dac = np.zeros((sweep_runs, freq_points))            # stores the DC reading via daq
det_out_lockin = np.zeros((sweep_runs, freq_points))          # stores the detector reading via lock-in red pitaya
true_signal = np.zeros((sweep_runs, freq_points))             # stores the true signal by removing true gain from lockin output
cont = np.zeros((sweep_runs, freq_points))                    # stores the lock-in/daq reading   # for contrast
freq_sweep, freq_step = np.linspace(start_freq, stop_freq, freq_points, retstep=True)                # frequency sweep points
total_time = time.strftime("%H:%M:%S", time.gmtime(wait_time*freq_points*sweep_runs)) # time that the full run will take
sweep_time = time.strftime("%H:%M:%S", time.gmtime(wait_time*freq_points))            # time that each sweep will take


# generating meta data

meta = smallGenMeta(rf=rf, laser_C1 = laser_C1, mw_gen = mw_gen, mw_amp = mw_amp, wait_time = wait_time, 
                n_points = freq_points, start_freq=start_freq, stop_freq=stop_freq, mod_freq=mod_freq, 
                time_constant=time_constant, sweep_runs=sweep_runs, current_preamp= current_preamp, 
                preamp_gain= preamp_gain, comments = comments, mw_switch = mw_switch, photodetector=photodetector,
                sample=sample,sampling_rate=sampling_rate_in, sweep_time = sweep_time, total_time = total_time,
                lockin_gain = lockin_gain, shift = delay_samples, samps_out=samps_out)

readDict(meta)  # printing meta data

General Filler : =====================General Settings===================
General Settings :
    Date Time : 2024-05-09T15:50:04.404929
    MW Source : Windfreak SynthHD Pro V1
    MW Switch : ZYSWA-2-50DR+
    MW PowerAmp : None
    Photo Diode : DET36A2
    Current Preamp : 1641
    Lock-In Amp : LIA/PS-1
    TTL Source : NI USB6343
    Sample : MPCVD-1
Measurement Filler : ===============Measurement Settings=================
Measurement Settings :
    Laser C1 : 8.75
    Preamp Gain (V per A) : 1000000
    MW Power (dBm) : 20.0
    Modulation Frequency (Hz) : 1470.59
    DC Sampling Rate (Hz) : 100000
    Time Constant (s) : 0.1
    Filter Slope dB/Octave : 6
    MW Frequency Sweep (MHz) : [2840. 2900.]
    No. of Sweep Points : 50
    No. of Sweep Runs : 1
    Lock-In Wait Time (s) : 1.0
    Lock-In Gain : 100
    Lock-In True Gain : 354.0
    Sweep time : 00:00:50
    Total time : 00:00:50
    MW Switch TTL ahead of Lock-In TTL by samples : 0
    Samples in one TTL cycle : 68
    

In [46]:
# RUN ONLY WHEN USING DAQ CARD

volt_max_dc = 1.5  # maximum dc voltage
task_read = nidaqmx.Task() # creating daq task instance
# task_read2 = nidaqmx.Task() # creating lockin reading task instance

task_read.ai_channels.add_ai_voltage_chan("Dev7/ai3", min_val=-1*volt_max_dc, max_val=volt_max_dc) # setting the analog read with apd #min and max voltage is set to -1.5V or 1.5V.
#volt_max_ac = 5
#task_read.ai_channels.add_ai_voltage_chan("Dev7/ai0", min_val=-1*volt_max_ac, max_val=volt_max_ac) # setting the other analog read with Lock in Output

# DAC sampling at 48 kHz means 960 samples every 20 ms which is the period of line frequency at 50 Hz
task_read.timing.cfg_samp_clk_timing(sampling_rate_in, source="", active_edge=Edge.RISING, sample_mode=AcquisitionType.FINITE, samps_per_chan=samps_per_chan) # setting the sampling rate and clock details

# task_read.start()
# print(task_read.read(number_of_samples_per_channel=5))
# task_read.stop()
# p = []
# for i in range(10):
#     task_read.start()
#     # print(task_read.read(number_of_samples_per_channel=960))
#     x = task_read.read(number_of_samples_per_channel=48000)
#     # time.sleep(0.00005)
#     p.append(np.round(np.mean(x),4))
#     task_read.stop()

# print(p)


In [ ]:
# CHECKING PL DC LEVEL AND DARK CURRENT

dc_pl = []

for i in range(20):
    try:
        task_read.start()
        plt.pause(plot_delay)
        p = task_read.read(number_of_samples_per_channel=samps_per_chan) # READS DATA FROM 2 CHANNELS
        task_read.stop()
        p = np.array(p[0])
        plt.ylabel('DC Reading (V)')
        plt.plot(i, np.mean(p), 'bo')
        dc_pl.append(np.mean(p))

    except KeyboardInterrupt:
        print('PL Check terminated')
        task_read.stop()
        break

print(np.mean(dc_pl))



In [47]:
# DATA AQUISITION SNIPPET

# Defining figure to display Lock-In, PL DC and Contrast vs MW Frequency on different plots

fig = plt.figure()
ax1 = fig.add_subplot(221, label = 'Lock-In Output')
ax2 = fig.add_subplot(222, label = 'DC Level')
ax3 = fig.add_subplot(223, label = 'True Signal')
ax4 = fig.add_subplot(224, label = 'Contrast')

ax1.set_xlim(start_freq, stop_freq)
ax2.set_xlim(start_freq, stop_freq)
ax3.set_xlim(start_freq, stop_freq)
ax4.set_xlim(start_freq, stop_freq)

ax1.set_xlabel('MW Frequency (Hz)')
ax2.set_xlabel('MW Frequency (Hz)')
ax3.set_xlabel('MW Frequency (Hz)')
ax4.set_xlabel('MW Frequency (Hz)')
ax1.set_ylabel('Lock-In Output (V)')
ax2.set_ylabel('DC Reading (V)')
ax3.set_ylabel('True Signal (V)')
ax4.set_ylabel('Contrast')

rf.enable = True     # opening the MW output

ao_task.start()      # starting the analog out task  # he is starting the analog out and the analog read task at different times
time.sleep(1)        # sleep for 1 second 

for i in range(sweep_runs):
    
    iq0.setup(input = 'in1', frequency = meas['Measurement Settings']['Red Pitaya Frequency Sweep'][i], amplitude = 0.0, gain = 0.0, quadrature_factor = 2.0,
              output_signal = 'quadrature', bandwidth = 1/meas['Measurement Settings']['Time Constant (s)'], acbandwidth = 50, phase = 0)
    iq1.setup(input = 'in2', frequency = meas['Measurement Settings']['Red Pitaya Frequency Sweep'][i], amplitude = 0.0, gain = 0.0, quadrature_factor = 2.0,
              output_signal = 'quadrature', bandwidth = 1/meas['Measurement Settings']['Time Constant (s)'], acbandwidth = 50, phase = 90)
    
    time.sleep(meta['Measurement Settings']['Lock-In Wait Time (s)'])  #to wait for the RC time constant
    
    for f in range(freq_points):

        try:

            rf.frequency = freq_sweep[f]
            plt.pause(plot_delay)

            task_read.start()
            x = task_read.read(number_of_samples_per_channel=samps_per_chan)
            task_read.stop()  # reading task is being started and stopped for every frequency value

            dc_read = np.array(x[0])  # gives the dc value from the APD
            
            dc_level_dac[i,f] = np.mean(dc_read)  # stores the current dc value in the list
            #det_out_lockin[i,f] = x[-1][-1] - lockin_offset  # stores the last (out of samples per channel) dc value generated at the output of the lock in amplifier
            det_out_lockin[i,f] = np.sqrt(s.voltage_in1**2 + s.voltage_in2**2)
            true_signal[i,f] = det_out_lockin[i,f]/meta['Measurement Settings']['Lock-In True Gain']  # check the value
            #cont[i,f] = true_signal[i,f]/dc_level_dac[i,f]
            cont[i,f] = 1 - (true_signal[i,f]/dc_level_dac[i,f])/(true_signal[0,f]/dc_level_dac[0])  # contrast based on the dc pl taken in previous block as percentage

            
            # LIVE PLOTTING

            ax4.plot(freq_sweep[f], cont[i, f], 'bo') 
            ax3.plot(freq_sweep[f], true_signal[i,f], 'go')     
            ax2.plot(freq_sweep[f], dc_level_dac[i, f], 'ro')
            ax1.plot(freq_sweep[f], det_out_lockin[i, f], 'ko')
        

        except KeyboardInterrupt:

            print('Loop terminated before sweep completion')
            termination_flag = 1
            task_read.stop()
            ao_task.stop()
            break


    if sweep_runs == 1: ax4.plot(freq_sweep, cont[i], 'r-')
    else: ax4.plot(freq_sweep, cont[i], label = f'Sweep: {i}')


ao_task.stop()


In [48]:
# STORING EVERYTHING IN AN HDF5 FILES


rf.enable = False                                           # closing the MW output
task_read.close()                                           # closing the DAQ in task
ao_task.stop()                                              # stoping the DAQ out task
ao_task.close()                                             # closing the DAQ out task
synth.close()                                               # closing MW control object


measured_data = {}

measured_data['MW Frequency (Hz)'] = freq_sweep
measured_data['Lock-In Output (V)'] = det_out_lockin
measured_data['True Lock-In AC Reading (v)'] = true_signal
measured_data['DAQ DC Reading (V)'] = dc_level_dac
measured_data['Contrast'] = cont

if termination_flag == 0:
    measured_data['Sweep Avg Lock-In Output (V)'] = np.mean(det_out_lockin, axis = 0)
    measured_data['Sweep Avg DAQ DC Reading (V)'] = np.mean(dc_level_dac, axis = 0)
    measured_data['Sweep Avg Contrast'] = np.mean(cont, axis = 0)
    measured_data['Sweep Avg True Lock-In AC Reading (V)'] = np.mean(true_signal, axis = 0)
else:
    measured_data['Sweep Avg Lock-In AC Reading (V)'] = 'No averaging because loop was force-stopped.'
    measured_data['Sweep Avg DAQ DC Reading (V)'] = 'No averaging because loop was force-stopped.'
    measured_data['Sweep Avg Contrast'] = 'No averaging because loop was force-stopped.'
    measured_data['Sweep Avg True Lock-In AC Reading (V)'] = 'No averaging because loop was force-stopped.'


filename = f"laserC1_{meta['Measurement Settings']['Laser C1']}"
filename = filename + f"_MWpower_{int(meta['Measurement Settings']['MW Power (dBm)'])}_dBm_TTL_shift_{meta['Measurement Settings']['MW Switch TTL ahead of Lock-In TTL by samples']}"
filename = filename + f"_TTL_cycle_samples_{meta['Measurement Settings']['Samples in one TTL cycle']}"
filename = filename + f"_Freqsweep_{float(meta['Measurement Settings']['MW Frequency Sweep (MHz)'][0])}-{float(meta['Measurement Settings']['MW Frequency Sweep (MHz)'][-1])}_MHz"
filename = filename + f"_mag0mA_lockin_gain_{float(meta['Measurement Settings']['Lock-In Gain'])}"
filename = filename + f"_tc_{float(meta['Measurement Settings']['Time Constant (s)'])}_modf_{meta['Measurement Settings']['Modulation Frequency (Hz)']}_Hz"
filename = filename + f"_avg{int(meta['Measurement Settings']['No. of Sweep Runs'])}_delay{np.round(float(meta['Measurement Settings']['Lock-In Wait Time (s)']),1)}s"
filename = filename + f"_filterslope_{int(meta['Measurement Settings']['Filter Slope dB/Octave'])}_dB"
filename = filename + f"_redpitaya_frequency_sweep_{float(meta['Measurement Settings']['Red Pitaya Frequency Sweep'])}_Hz"

trial = '_trial_1'
spot = ''
date = time.strftime("%Y%m%d") + '_'

filename = date + filename + spot + trial

res_file = pathname + '/' + filename + '.hdf5'

# CHECKING IF FILE EXISTS. IF IT DOES IT RENAMES TO FILENAME_COPY_2

check1 = File_Manipulator(pathname)

if check1.get_location(filename) is None:
    file1 = HDF5_file(res_file)
    print('New data file created')
else:
    copy_number = 2
    while check1.get_location(res_file[:-5] + f'_copy_{copy_number}.hdf5') is not None:
        copy_number += 1  # Increment copy number if file exists
    
    res_file = res_file[:-5] + f'_copy_{copy_number}.hdf5'
    file1 = HDF5_file(res_file)  # Create the new file with the updated name
    print(f'File exists. Creating a separate copy: {res_file}')


file1.writeHDF5(meta['General Settings'], 'General Settings', 'root')
file1.writeHDF5(meta['Measurement Settings'], 'Measurement Settings', 'root')
file1.writeHDF5(meta['Block_info'], 'Block Diagram', 'root')
file1.writeHDF5(measured_data, 'Measured Data', 'root')



File not found in the given pathname object.
New data file created


In [ ]:
synth.close()